# Step 2: Data Layer

Validates the schema in `src/schema.py` and the loader/normalize logic in `src/data_loader.py`.

This notebook uses small **synthetic** sample rows (not real league data) just to prove the
pipeline works end to end. Real historical season data still needs to be supplied (manual
export or API) before this can run against actual stats — see `ROADMAP.md` Step 1/2.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from data_loader import normalize_player_week, compute_fantasy_points
from schema import FULL_PPR_SCORING, LEAGUE_SETTINGS

In [2]:
# Synthetic sample data — shape matches PLAYERS_SCHEMA / WEEKLY_STATS_SCHEMA.
players_df = pd.DataFrame([
    {"player_id": 1, "name": "Sample RB", "position": "RB", "team": "AAA", "bye_week": 7},
    {"player_id": 2, "name": "Sample WR", "position": "WR", "team": "BBB", "bye_week": 9},
])

weekly_df = pd.DataFrame([
    {"player_id": 1, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 80, "rush_td": 1, "receptions": 3, "rec_yds": 20, "rec_td": 0,
     "fumbles_lost": 0, "two_pt_conversions": 0},
    {"player_id": 2, "season": 2025, "week": 1, "pass_yds": 0, "pass_td": 0, "pass_int": 0,
     "rush_yds": 0, "rush_td": 0, "receptions": 6, "rec_yds": 95, "rec_td": 1,
     "fumbles_lost": 0, "two_pt_conversions": 0},
])

players_df

,player_id,name,position,team,bye_week
0,1,Sample RB,RB,AAA,7
1,2,Sample WR,WR,BBB,9


In [3]:
player_week = normalize_player_week(players_df, weekly_df)
player_week

,player_id,season,week,pass_yds,pass_td,pass_int,rush_yds,rush_td,receptions,rec_yds,rec_td,fumbles_lost,two_pt_conversions,name,position,team,bye_week,fantasy_points
0,1,2025,1,0.0,0.0,0.0,80.0,1.0,3.0,20.0,0.0,0.0,0.0,Sample RB,RB,AAA,7,19.0
1,2,2025,1,0.0,0.0,0.0,0.0,0.0,6.0,95.0,1.0,0.0,0.0,Sample WR,WR,BBB,9,21.5


In [4]:
# Sanity check: Sample RB — 80 rush yds (8.0) + 1 rush TD (6.0) + 3 rec (3.0) + 20 rec yds (2.0) = 19.0
expected_rb_points = 80 * FULL_PPR_SCORING["rush_yds"] + 1 * FULL_PPR_SCORING["rush_td"] \
    + 3 * FULL_PPR_SCORING["receptions"] + 20 * FULL_PPR_SCORING["rec_yds"]
actual_rb_points = player_week.loc[player_week["player_id"] == 1, "fantasy_points"].item()
assert actual_rb_points == expected_rb_points, (actual_rb_points, expected_rb_points)
print("fantasy point math checks out:", actual_rb_points)

fantasy point math checks out: 19.0


`LEAGUE_SETTINGS` in `src/schema.py` currently holds standard Full PPR defaults (roster slots,
bench size, scoring weights) as a placeholder — still needs to be confirmed against the user's
actual league settings.

In [5]:
LEAGUE_SETTINGS

{'roster_slots': {'QB': 1,
  'RB': 2,
  'WR': 2,
  'TE': 1,
  'FLEX': 1,
  'K': 1,
  'DST': 1},
 'bench_size': 6,
 'scoring': {'pass_yds': 0.04,
  'pass_td': 4.0,
  'pass_int': -2.0,
  'rush_yds': 0.1,
  'rush_td': 6.0,
  'receptions': 1.0,
  'rec_yds': 0.1,
  'rec_td': 6.0,
  'fumbles_lost': -2.0,
  'two_pt_conversions': 2.0}}